In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install torch_geometric "scanpy==1.11.4"

## import

In [1]:
import torch

if torch.cuda.is_available():
    device = "cuda"
    print("GPU: ",torch.cuda.get_device_name(0))
else:
    device = "cpu"
    print("Using CPU")

GPU:  Tesla T4


In [2]:
##Working with google colab
import os
import sys

os.chdir("/content/drive/MyDrive/Thesis/Projects/Master_Thesis/Notebooks/SelfAttention")
cwd = os.getcwd()
print(cwd)

sys.path.append("../../")

/content/drive/MyDrive/Thesis/Projects/Master_Thesis/Notebooks/SelfAttention


In [3]:
import scanpy as sc
import numpy as np
import torch
from sklearn.preprocessing import Normalizer

import matplotlib.pyplot as plt

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['mathtext.fontset'] = 'dejavuserif'
plt.rcParams['font.family'] = 'arial'

pltkw = dict(bbox_inches='tight', transparent=True)

/usr/local/lib/python3.12/dist-packages/scanpy/_utils/__init__.py:33: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  from anndata import __version__ as anndata_version
/usr/local/lib/python3.12/dist-packages/scanpy/__init__.py:24: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):
/usr/local/lib/python3.12/dist-packages/scanpy/readwrite.py:16: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):


In [4]:
import SelfAttention as SA
import importlib

# Dataset

Founsation Models

In [5]:
adata = sc.read_h5ad("/content/drive/MyDrive/Thesis/Projects/Data/Breast_Cancer/FMs_3/UNI_adata.h5ad")

In [20]:
adata = sc.read_h5ad("/content/drive/MyDrive/Thesis/Projects/Data/Breast_Cancer/FMs_3/hoptimus_adata.h5ad")

In [ ]:
adata = sc.read_h5ad("/content/drive/MyDrive/Thesis/Projects/Data/Breast_Cancer/FMs_3/virchow_adata.h5ad")

Noise

In [5]:
adata = sc.read_h5ad("../../../Data/Breast_Cancer/ann_data.h5ad")

In [6]:
rng = np.random.default_rng(42)
ad = adata.copy()
ad.obsm["morpho"] = rng.standard_normal((167780, 500))

In [7]:
adata.obsm['p_Morpho_Embedding'] = ad.obsm['morpho'] - np.min(ad.obsm['morpho'])

Preparing Dataset

In [21]:
adata = SA.prep_adata(adata, norm=True, log1p=True)
data = SA.build_graph(adata, k=8, morph_key='p_Morpho_Embedding')

/usr/local/lib/python3.12/dist-packages/legacy_api_wrap/__init__.py:88: UserWarning: Some cells have zero counts
  return fn(*args_all, **kw)


In [7]:
print(data.x.shape)              # shape of each node
print(data.edge_index[:, :10])   # first 10 edges
print(data.edge_attr[:10])       # first 10 edge features

torch.Size([167780, 813])
tensor([[    0,     2,     0, 46964,     0,   112,     0,     4,     0,     5],
        [    2,     0, 46964,     0,   112,     0,     4,     0,     5,     0]])
tensor([ 5.7527,  5.7527,  6.9007,  6.9007,  8.1722,  8.1722,  8.4531,  8.4531,
        10.4105, 10.4105])


In [ ]:
importlib.reload(SA)
importlib.reload(SA.model)

<module 'SelfAttention.model' from '/content/drive/MyDrive/Thesis/Projects/Master_Thesis/Notebooks/SelfAttention/../../SelfAttention/model.py'>

In [13]:
import gc

gc.collect()
torch.cuda.empty_cache()

# Train

In [22]:
SA.set_random_seed(42)
model = SA.SpatialTransformerAE(
    in_dim=data.x.shape[1],
    gene_dim=data.gene_dim,
    hidden_dim=96,
    latent_dim=64,
    heads=4
)

## NOISE

In [11]:
# NOISE
SA.model.fit(model, data, max_epochs=10000, mask_ratio=0.8, stop_eps=1e-7, stop_tol=200, device=device)

Epoch 0 | Loss: 0.5199
Best : inf
Epoch 10 | Loss: 0.2902
Best : 0.2912
Epoch 20 | Loss: 0.2749
Best : 0.2768
Epoch 30 | Loss: 0.2532
Best : 0.2556
Epoch 40 | Loss: 0.2261
Best : 0.2289
Epoch 50 | Loss: 0.2033
Best : 0.2051
Epoch 60 | Loss: 0.1909
Best : 0.1917
Epoch 70 | Loss: 0.1860
Best : 0.1863
Epoch 80 | Loss: 0.1835
Best : 0.1837
Epoch 90 | Loss: 0.1817
Best : 0.1819
Epoch 100 | Loss: 0.1802
Best : 0.1804
Epoch 110 | Loss: 0.1791
Best : 0.1794
Epoch 120 | Loss: 0.1771
Best : 0.1773
Epoch 130 | Loss: 0.1752
Best : 0.1750
Epoch 140 | Loss: 0.1806
Best : 0.1721
Epoch 150 | Loss: 0.1691
Best : 0.1667
Epoch 160 | Loss: 0.1622
Best : 0.1649
Epoch 170 | Loss: 0.1616
Best : 0.1588
Epoch 180 | Loss: 0.1564
Best : 0.1551
Epoch 190 | Loss: 0.1531
Best : 0.1528
Epoch 200 | Loss: 0.1505
Best : 0.1509
Epoch 210 | Loss: 0.1507
Best : 0.1494
Epoch 220 | Loss: 0.1482
Best : 0.1485
Epoch 230 | Loss: 0.1475
Best : 0.1474
Epoch 240 | Loss: 0.1469
Best : 0.1470
Epoch 250 | Loss: 0.1464
Best : 0.1465


In [12]:
os.makedirs("saved_models", exist_ok=True)
model_dir = "saved_models"

torch.save(model.state_dict(), 'saved_models/breast_cancer_64_NOISE_0.8.pth')

## h_Optimus

In [23]:
# h_optimus
SA.model.fit(model, data, max_epochs=10000, mask_ratio=0.8, stop_eps=1e-7, stop_tol=200, device=device)

Epoch 0 | Loss: 0.3065
Best : inf
Epoch 10 | Loss: 0.2582
Best : 0.2640
Epoch 20 | Loss: 0.2054
Best : 0.2098
Epoch 30 | Loss: 0.1733
Best : 0.1756
Epoch 40 | Loss: 0.1572
Best : 0.1586
Epoch 50 | Loss: 0.1464
Best : 0.1472
Epoch 60 | Loss: 0.1405
Best : 0.1410
Epoch 70 | Loss: 0.1366
Best : 0.1370
Epoch 80 | Loss: 0.1332
Best : 0.1336
Epoch 90 | Loss: 0.1301
Best : 0.1304
Epoch 100 | Loss: 0.1272
Best : 0.1275
Epoch 110 | Loss: 0.1248
Best : 0.1251
Epoch 120 | Loss: 0.1228
Best : 0.1230
Epoch 130 | Loss: 0.1212
Best : 0.1214
Epoch 140 | Loss: 0.1200
Best : 0.1201
Epoch 150 | Loss: 0.1190
Best : 0.1191
Epoch 160 | Loss: 0.1182
Best : 0.1183
Epoch 170 | Loss: 0.1175
Best : 0.1176
Epoch 180 | Loss: 0.1168
Best : 0.1169
Epoch 190 | Loss: 0.1163
Best : 0.1164
Epoch 200 | Loss: 0.1159
Best : 0.1159
Epoch 210 | Loss: 0.1154
Best : 0.1155
Epoch 220 | Loss: 0.1150
Best : 0.1151
Epoch 230 | Loss: 0.1148
Best : 0.1148
Epoch 240 | Loss: 0.1145
Best : 0.1145
Epoch 250 | Loss: 0.1142
Best : 0.1142


In [24]:
os.makedirs("saved_models", exist_ok=True)
model_dir = "saved_models"

torch.save(model.state_dict(), 'saved_models/breast_cancer_64_hoptimus_0.8.pth')

## UNI

In [9]:
# UNI 0.8
SA.model.fit(model, data, max_epochs=10000, mask_ratio=0.8, stop_eps=1e-7, stop_tol=200, device=device)

Epoch 0 | Loss: 0.3004
Best : inf
Epoch 10 | Loss: 0.2583
Best : 0.2641
Epoch 20 | Loss: 0.2054
Best : 0.2100
Epoch 30 | Loss: 0.1733
Best : 0.1757
Epoch 40 | Loss: 0.1565
Best : 0.1578
Epoch 50 | Loss: 0.1460
Best : 0.1468
Epoch 60 | Loss: 0.1405
Best : 0.1410
Epoch 70 | Loss: 0.1368
Best : 0.1372
Epoch 80 | Loss: 0.1331
Best : 0.1335
Epoch 90 | Loss: 0.1297
Best : 0.1300
Epoch 100 | Loss: 0.1270
Best : 0.1273
Epoch 110 | Loss: 0.1250
Best : 0.1252
Epoch 120 | Loss: 0.1233
Best : 0.1235
Epoch 130 | Loss: 0.1219
Best : 0.1220
Epoch 140 | Loss: 0.1208
Best : 0.1210
Epoch 150 | Loss: 0.1198
Best : 0.1199
Epoch 160 | Loss: 0.1189
Best : 0.1190
Epoch 170 | Loss: 0.1181
Best : 0.1182
Epoch 180 | Loss: 0.1175
Best : 0.1175
Epoch 190 | Loss: 0.1169
Best : 0.1169
Epoch 200 | Loss: 0.1164
Best : 0.1164
Epoch 210 | Loss: 0.1160
Best : 0.1160
Epoch 220 | Loss: 0.1155
Best : 0.1156
Epoch 230 | Loss: 0.1152
Best : 0.1152
Epoch 240 | Loss: 0.1149
Best : 0.1149
Epoch 250 | Loss: 0.1146
Best : 0.1147


In [10]:
os.makedirs("saved_models", exist_ok=True)
model_dir = "saved_models"

torch.save(model.state_dict(), 'saved_models/Breast_Cancer_64_UNI_0.8.pth')

In [11]:
SA.set_random_seed(42)
model = SA.SpatialTransformerAE(
    in_dim=data.x.shape[1],
    gene_dim=data.gene_dim,
    hidden_dim=96,
    latent_dim=64,
    heads=4
)

In [12]:
# UNI 0.5
SA.model.fit(model, data, max_epochs=10000, mask_ratio=0.5, stop_eps=1e-7, stop_tol=200, device=device)

Epoch 0 | Loss: 0.3008
Best : inf
Epoch 10 | Loss: 0.2565
Best : 0.2623
Epoch 20 | Loss: 0.2029
Best : 0.2075
Epoch 30 | Loss: 0.1701
Best : 0.1724
Epoch 40 | Loss: 0.1530
Best : 0.1545
Epoch 50 | Loss: 0.1421
Best : 0.1429
Epoch 60 | Loss: 0.1355
Best : 0.1361
Epoch 70 | Loss: 0.1297
Best : 0.1303
Epoch 80 | Loss: 0.1245
Best : 0.1250
Epoch 90 | Loss: 0.1204
Best : 0.1208
Epoch 100 | Loss: 0.1174
Best : 0.1176
Epoch 110 | Loss: 0.1152
Best : 0.1155
Epoch 120 | Loss: 0.1138
Best : 0.1139
Epoch 130 | Loss: 0.1126
Best : 0.1127
Epoch 140 | Loss: 0.1115
Best : 0.1117
Epoch 150 | Loss: 0.1105
Best : 0.1106
Epoch 160 | Loss: 0.1096
Best : 0.1098
Epoch 170 | Loss: 0.1087
Best : 0.1088
Epoch 180 | Loss: 0.1079
Best : 0.1080
Epoch 190 | Loss: 0.1073
Best : 0.1073
Epoch 200 | Loss: 0.1066
Best : 0.1067
Epoch 210 | Loss: 0.1069
Best : 0.1063
Epoch 220 | Loss: 0.1056
Best : 0.1057
Epoch 230 | Loss: 0.1052
Best : 0.1052
Epoch 240 | Loss: 0.1048
Best : 0.1047
Epoch 250 | Loss: 0.1043
Best : 0.1043


In [13]:
os.makedirs("saved_models", exist_ok=True)
model_dir = "saved_models"

torch.save(model.state_dict(), 'saved_models/Breast_Cancer_64_UNI_0.5.pth')

In [14]:
SA.set_random_seed(42)
model = SA.SpatialTransformerAE(
    in_dim=data.x.shape[1],
    gene_dim=data.gene_dim,
    hidden_dim=96,
    latent_dim=64,
    heads=4
)

In [15]:
# UNI 0.2
SA.model.fit(model, data, max_epochs=10000, mask_ratio=0.2, stop_eps=1e-7, stop_tol=200, device=device)

Epoch 0 | Loss: 0.3012
Best : inf
Epoch 10 | Loss: 0.2553
Best : 0.2610
Epoch 20 | Loss: 0.2012
Best : 0.2059
Epoch 30 | Loss: 0.1678
Best : 0.1702
Epoch 40 | Loss: 0.1505
Best : 0.1519
Epoch 50 | Loss: 0.1393
Best : 0.1401
Epoch 60 | Loss: 0.1320
Best : 0.1326
Epoch 70 | Loss: 0.1253
Best : 0.1259
Epoch 80 | Loss: 0.1193
Best : 0.1198
Epoch 90 | Loss: 0.1150
Best : 0.1154
Epoch 100 | Loss: 0.1117
Best : 0.1120
Epoch 110 | Loss: 0.1093
Best : 0.1095
Epoch 120 | Loss: 0.1073
Best : 0.1074
Epoch 130 | Loss: 0.1053
Best : 0.1055
Epoch 140 | Loss: 0.1035
Best : 0.1036
Epoch 150 | Loss: 0.1023
Best : 0.1023
Epoch 160 | Loss: 0.1005
Best : 0.1005
Epoch 170 | Loss: 0.0993
Best : 0.0994
Epoch 180 | Loss: 0.0983
Best : 0.0984
Epoch 190 | Loss: 0.0974
Best : 0.0976
Epoch 200 | Loss: 0.0966
Best : 0.0967
Epoch 210 | Loss: 0.0959
Best : 0.0959
Epoch 220 | Loss: 0.0953
Best : 0.0952
Epoch 230 | Loss: 0.0946
Best : 0.0946
Epoch 240 | Loss: 0.0940
Best : 0.0941
Epoch 250 | Loss: 0.0934
Best : 0.0935


In [16]:
os.makedirs("saved_models", exist_ok=True)
model_dir = "saved_models"

torch.save(model.state_dict(), 'saved_models/Breast_Cancer_64_UNI_0.2.pth')

In [17]:
SA.set_random_seed(42)
model = SA.SpatialTransformerAE(
    in_dim=data.x.shape[1],
    gene_dim=data.gene_dim,
    hidden_dim=96,
    latent_dim=64,
    heads=4
)

In [18]:
# UNI 1.0
SA.model.fit(model, data, max_epochs=10000, mask_ratio=1.0, stop_eps=1e-7, stop_tol=200, device=device)

Epoch 0 | Loss: 0.3001
Best : inf
Epoch 10 | Loss: 0.2600
Best : 0.2657
Epoch 20 | Loss: 0.2078
Best : 0.2124
Epoch 30 | Loss: 0.1760
Best : 0.1783
Epoch 40 | Loss: 0.1596
Best : 0.1610
Epoch 50 | Loss: 0.1498
Best : 0.1505
Epoch 60 | Loss: 0.1448
Best : 0.1452
Epoch 70 | Loss: 0.1416
Best : 0.1419
Epoch 80 | Loss: 0.1390
Best : 0.1392
Epoch 90 | Loss: 0.1367
Best : 0.1369
Epoch 100 | Loss: 0.1351
Best : 0.1352
Epoch 110 | Loss: 0.1338
Best : 0.1339
Epoch 120 | Loss: 0.1329
Best : 0.1330
Epoch 130 | Loss: 0.1324
Best : 0.1324
Epoch 140 | Loss: 0.1317
Best : 0.1316
Epoch 150 | Loss: 0.1310
Best : 0.1310
Epoch 160 | Loss: 0.1304
Best : 0.1305
Epoch 170 | Loss: 0.1299
Best : 0.1300
Epoch 180 | Loss: 0.1295
Best : 0.1295
Epoch 190 | Loss: 0.1291
Best : 0.1292
Epoch 200 | Loss: 0.1287
Best : 0.1287
Epoch 210 | Loss: 0.1282
Best : 0.1283
Epoch 220 | Loss: 0.1279
Best : 0.1280
Epoch 230 | Loss: 0.1276
Best : 0.1276
Epoch 240 | Loss: 0.1273
Best : 0.1274
Epoch 250 | Loss: 0.1270
Best : 0.1270


In [19]:
os.makedirs("saved_models", exist_ok=True)
model_dir = "saved_models"

torch.save(model.state_dict(), 'saved_models/Breast_Cancer_64_UNI_1.0.pth')

## virchow

In [ ]:
# virchow
SA.model.fit(model, data, max_epochs=10000, mask_ratio=0.8, stop_eps=1e-7, stop_tol=200, device=device)

Epoch 0 | Loss: 0.3286
Best : inf
Epoch 10 | Loss: 0.2877
Best : 0.2903
Epoch 20 | Loss: 0.2579
Best : 0.2612
Epoch 30 | Loss: 0.2244
Best : 0.2276
Epoch 40 | Loss: 0.1956
Best : 0.1981
Epoch 50 | Loss: 0.1763
Best : 0.1779
Epoch 60 | Loss: 0.1645
Best : 0.1655
Epoch 70 | Loss: 0.1563
Best : 0.1570
Epoch 80 | Loss: 0.1506
Best : 0.1511
Epoch 90 | Loss: 0.1464
Best : 0.1468
Epoch 100 | Loss: 0.1434
Best : 0.1436
Epoch 110 | Loss: 0.1411
Best : 0.1413
Epoch 120 | Loss: 0.1392
Best : 0.1393
Epoch 130 | Loss: 0.1374
Best : 0.1376
Epoch 140 | Loss: 0.1358
Best : 0.1360
Epoch 150 | Loss: 0.1343
Best : 0.1345
Epoch 160 | Loss: 0.1329
Best : 0.1331
Epoch 170 | Loss: 0.1316
Best : 0.1317
Epoch 180 | Loss: 0.1303
Best : 0.1304
Epoch 190 | Loss: 0.1290
Best : 0.1291
Epoch 200 | Loss: 0.1276
Best : 0.1277
Epoch 210 | Loss: 0.1262
Best : 0.1263
Epoch 220 | Loss: 0.1248
Best : 0.1250
Epoch 230 | Loss: 0.1236
Best : 0.1237
Epoch 240 | Loss: 0.1225
Best : 0.1226
Epoch 250 | Loss: 0.1217
Best : 0.1218


In [ ]:
os.makedirs("saved_models", exist_ok=True)
model_dir = "saved_models"

torch.save(model.state_dict(), 'saved_models/Breast_Cancer_32_virchow_0.8.pth')